# Hull Tactical - Market Prediction

**notebookタイトル**: EDA which makes sense for the Hull Tactical Market Prediction
**原著者**: ambrosm（Kaggle Notebooks Grandmaster）
**元notebookへのリンク**: https://www.kaggle.com/code/ambrosm/htmp-eda-which-makes-sense
**投票数**: 335（Gold medal）。Best Scoreでの明示的な提出スコアは付いていない（EDA・検証手法の解説が主目的のnotebookのため）。

**手法の概要**: このコンペのデータ構造（train/testの特殊な列、S&P 500の実データとの対応関係）を丁寧に説明した上で、コンペの採点関数（配分戦略のリターンとボラティリティのトレードオフを評価する独自指標）を分析し、シンプルな配分戦略とクロスバリデーションの正しいやり方を示す、非常に教育的なEDAノートブック。

> これは学習目的の解説付き写しです。元のコードセルは可能な限りそのまま保持していますが、**Kaggle上のレンダリング済みページからテキストとして抽出したためインデント（字下げ）情報が失われており**、このノートブックのインデントは処理内容の論理構造から筆者（本ルーティンを実行したAIエージェント）が再構成したものです。また、**元notebookにはグラフ描画のみを行うコードセルが複数個所「折りたたみ（hidden code）」表示になっており、その部分のソースコード自体がページ上に存在しませんでした**。該当箇所はコードを省略のうえ、生成される図の内容を本文の説明から要約したMarkdownで補っています（コードを捏造していません）。実行はしていません（出力セルは元notebookに表示されていた数値をそのまま引用しています）。

## 評価指標について

- **タスク**: 毎日、資産の何倍（0〜2倍、2倍はレバレッジ）をS&P 500に配分するかを決める配分（アロケーション）タスク。単純な数値予測ではなく、意思決定（ポジションサイジング）そのものが評価対象になる。
- **評価指標**: 本notebookの解説によれば、コンペ独自のスコア関数は「リターン」と「ボラティリティ」の両方を見る。具体的には、
  - ボラティリティが市場ボラティリティの1.2倍を超えると「ボラティリティ・ペナルティ」（超過分の差）が科される。
  - 配分が1.0を下回ると「リターン・ペナルティ」（1.0からの下振れの2乗）が科される。
  - 定数配分で最もスコアが高いのは0.8前後、Sharpe比はどの定数戦略でもおおよそ0.4〜0.5の範囲で大差ないという実験結果を提示している。
- **特性と指標選定の考察**: 単純な予測誤差（MSEなど）ではなく、リスク調整後リターンに近い設計になっているのは、金融の実務で「当たっているかどうか」だけでなく「どれだけの損失リスクを取ってその精度を得たか」が重要だからだと考えられる。また、著者は「リーダーボードのスコアはこのコンペでは完全にミスリーディング」と明言しており（Codeタブの一覧で見た `17.507` のような不自然に高いスコアの数々は、この警告と整合する）、公開リーダーボードのスコアを信用せず、自前で正しいクロスバリデーションを組む必要性を強調している。
- **選んだ手法がどう指標を最適化しているか**: 単純な「ボラティリティが低い時は多めに張る」戦略で0.53のスコアを達成し、ボラティリティ・ペナルティをほぼ最適（1.00）にできることを示す一方、リターン面の工夫が欠けている点を明示的に指摘し、「この2つを組み合わせれば勝てる」という次のステップを示唆している。


### データの特殊な列について（What / Why）

**何をしているか**: `train` には9021営業日分の履歴データがあり、目的変数 `forward_returns`（翌日にS&P 500に投資した場合のリターン）と、リスクフリーレート、両者を組み合わせた `market_forward_excess_returns`（超過リターン）が含まれる。`test` には「予測を行う時点でまだ分かっていない」情報は含まれず、代わりに1日遅れの `lagged_forward_returns` 列などが入っている。

**なぜそうするのか**: 時系列コンペでは「その時点で本当に知りえた情報は何か」を正確に区別しないと、未来の情報を使ってしまう典型的なリーク（データ漏洩）が起きる。`test` に生の `forward_returns` ではなく1日遅れの `lagged_` 列しか無いのは、本番の逐次評価（Kaggleがテストデータを1行ずつモデルに見せていく方式）を模した安全な設計になっている。


In [ ]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train[['date_id', 'forward_returns', 'risk_free_rate', 'market_forward_excess_returns']]


In [ ]:
test = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv')
test[['date_id', 'is_scored', 'lagged_forward_returns', 'lagged_risk_free_rate', 'lagged_market_forward_excess_returns']]


### 重複列の確認（What / Why）

**何をしているか**: 特徴量 `D1` と `D2` がtrainデータ内で完全に一致するか確認する。

**なぜそうするのか**: 完全に同じ列が2つあれば片方を削除して良さそうに見えるが、著者は「非公開のテストセットでは違う値になるかもしれないので、今は消さない」と慎重な判断をしている。EDAで見つけた「一見無駄に見える列」でも、テスト時の挙動が保証されていない限り安易に削除しないという実務的な注意点。


In [ ]:
(train.D1 == train.D2).mean()


### タイムライン可視化（What / Why、一部は元notebookでも折りたたみ表示）

**何をしているか**: 累積リターンの推移をプロットするとS&P 500の1993〜2025年の実際の値動き（macrotrends.netのチャートなど）に近い形になることを確認する（このプロットセル自体は元notebookで折りたたまれておりコードは非表示）。また `forward_returns` は常に-4%〜+4%の範囲に収まり、ボラティリティが高い時期と低い時期が交互に現れることも可視化で確認している（このプロットセルも非表示）。

**なぜそうするのか**: 時系列データはまず可視化して全体の傾向・周期性・異常値を把握するのが定石。この作業を通じて、`train.csv` の `date_id` が実際にはS&P 500の連続した営業日に対応しているらしいという仮説を立てる（次のセルで裏付けを取る）。


### 外部データセット（実際のカレンダー日付）との対応づけ（What / Why）

**何をしているか**: 著者が別途公開した補助データセット「S&P Historical Data for Hull Tactical Competition」を読み込む。これは `date_id` と実際のカレンダー日付を対応づけるデータで、S&P指数版（`sp-historical.csv`）とSPY ETF版（`spy-historical.csv`、配当を考慮したリターン計算つき）の2種類がある。

**なぜそうするのか**: コンペの `train.csv` には実際の日付が含まれていないため、曜日や季節性を直接特徴量にできない。実データとの対応づけができれば、「何曜日か」「何月か」といった外部知識を安全に取り込める（提供データだけでは分からない情報を、ライセンス上問題のない公開データで補う典型的なテクニック）。


In [ ]:
extra_dataset_sp = pd.read_csv('/kaggle/input/s-and-p-historical-data-for-hull-tactical-competition/sp-historical.csv', parse_dates=['Date'])
extra_dataset_spy = pd.read_csv('/kaggle/input/s-and-p-historical-data-for-hull-tactical-competition/spy-historical.csv', parse_dates=['Date'], index_col='date_id')
display(extra_dataset_spy.head())


### スコアリング関数の理解と可視化（What / Why、一部は元notebookでも折りたたみ表示）

**何をしているか**: コンペのデータとYahoo Financeの日次リターンを散布図で比較して高い相関を確認（この比較図のコードセルは元notebookで非表示）。曜日ごとの平均リターンを集計すると水曜・木曜のリターンが月・火・金より明確に小さいという奇妙なパターンを発見（有意かどうかはコメント欄で読者に問いかけている、可視化コードは非表示）。リスクフリーレートは年率0〜8%相当（1日あたり0〜0.03%）で推移していることを確認する（可視化コードは非表示）。市場超過リターン `market_forward_excess_returns` のグラフは以下のIn[11]で明示的にコードが示されている。

**なぜそうするのか**: 提供データの品質・妥当性を外部データと突き合わせて検証することは、後続のモデリングで「変な特徴量に惑わされない」ために重要。曜日パターンのような一見面白い発見も、「統計的に有意か」を鵜呑みにせず疑いの目で見る姿勢（本文でも読者に意見を求めている）は、過学習を避ける上で参考になる。


In [ ]:
plt.figure(figsize=(15, 4))
plt.scatter(train.date_id, train.market_forward_excess_returns, s=1)
plt.title('market_forward_excess_returns')
plt.xlabel('date_id')
plt.show()


### 配分（アロケーション）タスクとスコア関数の仕組み（What / Why）

**何をしているか**: このコンペのタスクを整理する。毎日、資産の0〜2倍をS&P 500に配分する意思決定を行い、0は「投資せずリスクフリーレートのみ得る」、1は「全額投資」、2は「自己資金と同額を借りてレバレッジ投資」を意味する。定数配分（毎日同じ比率で投資する単純戦略）でスコアを計算すると、0.8付近が最もスコアが高く、Sharpe比はどの定数戦略でも0.4〜0.5程度でほぼ変わらないことを可視化で確認する（このプロットセルも元notebookで非表示）。1.2を超える配分にはボラティリティ・ペナルティ、1.0未満の配分にはリターン・ペナルティが科されることも図から読み取れる。

**なぜそうするのか**: 単純な「予測精度」の最適化ではなく、「どれだけリスクを取ってどれだけのリターンを得たか」のバランスを取る意思決定問題であることを明確に理解しないと、モデルの予測をそのまま配分に使うような誤った実装をしてしまう。この整理があるからこそ、後続の「ボラティリティに応じて配分を変える」戦略の妥当性が見えてくる。


### シンプルな配分戦略: 直近のボラティリティに応じて配分を変える（What / Why）

**何をしているか**: 直近80日間のリターンの標準偏差（ボラティリティ）を計算し、ボラティリティが低い時は配分を増やし、高い時は減らすという単純な線形ルールで配分を決める。パラメータ（窓幅80、係数45、オフセット0.006）は試行錯誤で決めたと明記されている。このルールでtrainデータ全体に対してスコアを計算すると0.53となり、定数配分より良いスコアが出ることを確認する。

**なぜそうするのか**: 前セルで「ボラティリティが低い時期と高い時期が交互に来る」ことを確認済みなので、その周期性を利用して配分を動的に調整するのは自然な発想。ボラティリティ・ペナルティをほぼ最適（1.00）にできる一方、リターン・ペナルティは1.01とわずかに残っている点から、「このシンプルな戦略はボラティリティしか見ておらず、期待リターンの予測を組み合わせればさらに伸びる」という次の課題が見えてくる（本文でも明言されている）。


In [ ]:
# The parameters w, f and o were determined by trial and error
w = 80
f = 45
o = 0.006
train['last_week_volatility'] = train['forward_returns'].rolling(window=w).std().shift(1).fillna(0.04)
intermediate_res = []
submission = pd.DataFrame({'prediction': 1 + o * f - train['last_week_volatility'] * f}, index=train.index).clip(0, 2)
print(f"Score: {score(train.copy(), submission, '')}")
print(f"Volatility penalty: {intermediate_res[-1][3]:.2f} (1.00 is the optimum)")
print(f"Return penalty: {intermediate_res[-1][4]:.2f} (1.00 is the optimum)")


### 配分と実際のリターンの関係を可視化（What / Why）

**何をしているか**: 上のシンプルな戦略が実際にどう配分しているか（ボラティリティが高い時期に配分を下げているか）を、`forward_returns` の推移と並べてプロットする。

**なぜそうするのか**: 数値のスコアだけでなく、実際の挙動を目で確認することで「意図した通りに動いているか」「不自然な挙動をしていないか」を検証できる。これは機械学習全般で重要な「予測結果の可視化によるデバッグ」の実践例。


In [ ]:
_, ax = plt.subplots(2, 1, sharex=True, figsize=(15, 8))
ax[0].scatter(train.date_id, submission.prediction, s=1, color='m')
ax[0].set_title('allocation with the simple allocation strategy')
ax[0].set_xlabel('date_id')

ax[1].scatter(train.date_id, train.forward_returns, s=1)
ax[1].set_title('forward_returns')
ax[1].set_xlabel('date_id')
plt.show()


### 正しいクロスバリデーションの実装（What / Why）

**何をしているか**: `cross_validate` 関数を定義する。データが小さい（9021行）ため単一の train/test 分割では簡単に過学習してしまうとして、時系列に沿って複数foldに分ける（`min_train_size` 以降を `test_size` ごとに区切っていく、直近から過去に向かって順にfoldを作る）。各foldで、`test` 側には（本番と同様に）ラグ付きの目的変数しか渡さない設計にしている。`allocation_model` はscikit-learn風の `fit`/`predict` インターフェースを持つオブジェクトとして渡す。

**なぜそうするのか**: 時系列データでは通常のk-fold（ランダムシャッフル）クロスバリデーションを使うと未来のデータで過去を予測してしまうリークが起きる。本関数は必ず「過去のデータで学習→未来のデータでテスト」という順序を守るように設計されている。また「リーダーボードのスコアはこのコンペでは完全にミスリーディング」と明言されている通り、公開リーダーボードを信用せず、自前のクロスバリデーションこそが正しい意思決定の拠り所になる、という強いメッセージがこの関数の存在理由。


In [ ]:
import polars as pl

score_list_dict = {}

def cross_validate(allocation_model, label='', min_train_size=1500, test_size=120):
    """Print the validation score for allocation_model and the given train-test split.

    Parameters:
    allocation_model: object with methods fit and predict
    label: the name of the model
    min_train_size: minimum number of samples for training, taken from the beginning of the training dataset
    test_size: number of samples for testing, taken from the time period immediately after the training
    """
    # Read the training dataset
    train = pl.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

    oof = np.full(len(train), np.nan)
    score_list = []
    for fold, test_start in enumerate(range(len(train) - test_size, min_train_size, - test_size)):

        # Split into train and test
        # test has lagged targets rather than current targets
        test_preliminary = train.slice(test_start, test_size)
        solution = pd.DataFrame(test_preliminary.select('forward_returns', 'risk_free_rate'),
                                 columns=['forward_returns', 'risk_free_rate'])
        lagged = train.slice(test_start - 1, test_size)
        test = (
            test_preliminary
            .drop('forward_returns', 'risk_free_rate', 'market_forward_excess_returns')
            .with_columns(
                lagged.get_column('forward_returns').alias('lagged_forward_returns'),
                lagged.get_column('risk_free_rate').alias('lagged_risk_free_rate'),
                lagged.get_column('market_forward_excess_returns').alias('lagged_market_forward_excess_returns'),
            )
        )
        del test_preliminary
        train1 = train.slice(0, test_start)
        # print(train1)
        # print(test)

        # Fit the model
        allocation_model.fit(train1)

        # Predict for validation
        assert test['date_id'].is_sorted()
        allocation_list = []
        batch_ids = test['date_id'].unique(maintain_order=True).to_list()
        for batch_id in batch_ids:
            test_batch = test.filter(pl.col('date_id') == batch_id)
            allocation_list.append(allocation_model.predict(test_batch))

        # Score the validation predictions of this fold
        allocation_list = np.array(allocation_list, dtype=np.float32)
        submission = pl.DataFrame({'prediction': allocation_list})
        validation_score = score(solution, submission, '')
        vol_penalty = intermediate_res[-1][3]
        return_penalty = intermediate_res[-1][4]
        if fold <= 2 or fold >= 59:
            print(f"# Fold {fold:2} train(:{test_start:4}) test({test_start:4}:{test_start+test_size:4}) val_score: {validation_score:6.3f} {vol_penalty=:.2f} {return_penalty=:.2f}")
        if fold == 2:
            print('...')
        oof[test_start:test_start+test_size] = allocation_list
        score_list.append(validation_score)

    # Score the validation predictions overall
    print(f"{Fore.RED}# Average validation score: {np.array(score_list).mean():6.3f}", end=' ')
    solution = pd.DataFrame(train.select('forward_returns', 'risk_free_rate'),
                             columns=['forward_returns', 'risk_free_rate'])
    submission = pl.DataFrame({'prediction': oof[np.isfinite(oof)]})
    validation_score = score(solution[np.isfinite(oof)].copy(), submission, '')
    print(f"Overall validation score: {validation_score:6.3f} {label}{Style.RESET_ALL}")
    score_list_dict[label] = score_list

    # Show a histogram of the predictions
    plt.figure(figsize=(6, 2))
    plt.hist(oof, bins=np.linspace(0, 2, 81), density=True, color='c')
    plt.title(f'Allocation histogram of {label}')
    plt.gca().get_yaxis().set_visible(False)
    plt.show()


### 2つのモデルをクロスバリデーションで比較（What / Why）

**何をしているか**: `ConstantAllocationModel`（常に一定の配分を返すだけのベースライン）と `LinearAllocationModel`（14個の特徴量からRidge回帰でリターンを予測し、予測値を150倍して1を足した値をクリップして配分にする線形モデル）を定義し、`cross_validate` で評価する。`LinearAllocationModel` はボラティリティを一切考慮していない（本文でも明記）。全foldの平均スコアと、全体を通した総合スコアの両方を出力する。

**なぜそうするのか**: ベースラインとの比較なしに「良いモデル」は語れないため、まず単純な定数戦略と線形モデルを比較する。出力される「fold平均スコア」と「全体総合スコア」の乖離が大きい点（例: LinearAllocationModelはfold平均0.902だが全体では0.428）は、100日程度の短い期間のボラティリティが35年間全体のボラティリティより小さいために生じる指標の性質であり、著者はこれも解説している。この気づきは「クロスバリデーションのスコアの見せ方一つで印象が大きく変わる」ことを示す重要な注意点。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

class ConstantAllocationModel:
    """A model which ignores all features and always predicts the allocation given to __init__"""
    def __init__(self, constant_allocation):
        self.constant_allocation = constant_allocation

    def fit(self, train: pl.DataFrame):
        pass

    def predict(self, test: pl.DataFrame) -> float:
        """Return the optimal allocation between 0.0 and 2.0.

        Parameter:
        test: polars DataFrame with a single row (which is ignored)
        """
        return self.constant_allocation

class LinearAllocationModel:
    """A linear model which predicts returns based on a feature subset and
    computes the allocation as a function of the predicted returns.

    This model neglects the volatility."""
    vars_to_keep = ['M3', 'V2', 'P13', 'E14', 'S5', 'S4', 'S11',
                     'P10', 'P8', 'E3', 'P6', 'E20', 'P11', 'M12']

    def __init__(self):
        pass

    def fit(self, train: pl.DataFrame):
        train = train.select(type(self).vars_to_keep + ['forward_returns', 'risk_free_rate', 'market_forward_excess_returns'])
        train = train.slice(1006)  # the first 1006 rows have too many missing values
        self.prediction_model = make_pipeline(SimpleImputer(), Ridge())
        self.prediction_model.fit(np.array(train.select(type(self).vars_to_keep)),
                                   train.get_column('forward_returns'))

    def predict(self, test: pl.DataFrame) -> float:
        """Return the optimal allocation between 0.0 and 2.0.

        Parameter:
        test: polars DataFrame with a single row
        """
        predicted_returns = self.prediction_model.predict(
            np.array(test.select(type(self).vars_to_keep))
        )
        predicted_returns = predicted_returns.item()
        allocation = predicted_returns * 150 + 1
        return np.clip(allocation, 0, 2)

test_size = 200
cross_validate(ConstantAllocationModel(0.8), label='ConstantAllocationModel 0.8')
cross_validate(ConstantAllocationModel(1.2), label='ConstantAllocationModel 1.2')
cross_validate(LinearAllocationModel(), label='LinearAllocationModel')
# Average validation score: 0.607 Overall validation score: 0.451 ConstantAllocationModel
# Average validation score: 0.902 Overall validation score: 0.475 LinearAllocationModel

# Fold  0 train(:8901) test(8901:9021) val_score:  2.153 vol_penalty=1.00 return_penalty=1.56
# Fold  1 train(:8781) test(8781:8901) val_score: -1.130 vol_penalty=1.00 return_penalty=1.00
# Fold  2 train(:8661) test(8661:8781) val_score:  1.378 vol_penalty=1.00 return_penalty=1.18
# ...
# Fold 61 train(:1581) test(1581:1701) val_score:  0.519 vol_penalty=1.00 return_penalty=1.01
# Average validation score: 0.611 Overall validation score: 0.455 ConstantAllocationModel 0.8

# (ConstantAllocationModel 1.2, LinearAllocationModel の詳細foldログは元notebookと同様に省略)
# Average validation score: 0.807 Overall validation score: 0.428 ConstantAllocationModel 1.2
# Average validation score: 0.866 Overall validation score: 0.482 LinearAllocationModel


### まとめ（元notebookの結び）

fold単位のスコアを可視化すると（このグラフのコードセルも元notebookで非表示）、ほとんどのfoldで `LinearAllocationModel` が `ConstantAllocationModel` を上回ることが分かる。ただし検証スコアは選ぶ期間によって大きく変動する（金融市場には良い年も悪い年もある、というのが著者の言葉）。

このタイムシリーズコンペは、単に誤差最小化で目的変数を当てるだけの問題ではない。期待リターンの予測に加えて、その予測の不確実性（ボラティリティ）を定量化し、両者を踏まえて資金配分を最適化する必要がある——というのが著者の結論。
